## Look for PII with Regex

In [1]:
import re

https://docs.python.org/3/library/re.html#re.search

In [2]:
def check_for_PII(text):
    ssn_pattern = r"\b\d{3}-\d{2}-\d{4}\b"
    return bool(re.search(ssn_pattern, text))

In [4]:
?bool

Init signature: bool(self, /, *args, **kwargs)
Docstring:     
bool(x) -> bool

Returns True when the argument x is true, False otherwise.
The builtins True and False are the only two instances of the class bool.
The class bool is a subclass of the class int, and cannot be subclassed.
Type:           type
Subclasses:     

In [3]:
check_for_PII("My SSN is 123-45-6789")

True

In [4]:
check_for_PII("No number here!")

False

In [5]:
import openai

## get_LLM_response
Scoped API key to Restricted -> Model capabilities -> Request.

In [9]:
def get_LLM_response(prompt):
    model_engine = "davinci-002"
    response = openai.Completion.create(
        engine=model_engine,
        prompt=prompt,
        max_tokens=100,
    )
    return response.choices[0].text.strip()

See https://developers.openai.com/api/docs/deprecations#instructgpt-models for when `text-davinci-002` was deprecated.

Also, see https://developers.openai.com/api/docs/pricing for current models and pricing.

In [10]:
get_LLM_response("Tell me your thoughts on universal healthcare.")

'– Clare De Castillon\n\n***\n\nStrange Rain\n\nlines like an empty map\n\nwhy can’t i find you –\n\nsea reads like a mirror\n\nnights dead without a name\n\nfloating gone like dust\n\nand the seasons fluttered and blocked my top off\n\nand you windows shuddered with love for a mouth\n\nlost was the line\n\nas if disappeared\n\ntime\xa0and\xa0and\xa0and weight\xa0and\n\nand\xa0and why\n\ni haven’t slept in months\n\ntumbling blind'

## Evaluate for Toxicity

Scoped API key to Restricted -> Model capabilities -> Moderations -> Request.

In [6]:
def check_toxicity(text):
    """
    Check the toxicity of a text using OpenAI's Moderation API.

    Args:
        text: The text to check for toxicity.

    Returns:
        A toxicity score between 0 and 1, where a higher score indicates a
        higher probability of the text being toxic.
    """
    response = openai.Moderation.create(input=text)
    return response["results"][0]["flagged"]


In [7]:
check_toxicity("You are stupid.")

True

In [8]:
check_toxicity("Hello world!")

False

## Sanitize for Safety
https://docs.python.org/3/library/html.html#html.escape

In [1]:
import html

In [3]:
def sanitize_output(text):
    return html.escape(text)


In [4]:
sanitize_output("<script>alert(0)</script>")

'&lt;script&gt;alert(0)&lt;/script&gt;'

In [5]:
sanitize_output("foobar")

'foobar'

## Link Filters to LLM

In [12]:
def log_results(prompt, llm_output, is_safe):
    with open("llm_safety_log.txt", "a") as log_file:
        log_file.write(f"Prompt: {prompt}\n")
        log_file.write(f"LLM Output: {llm_output}\n")
        log_file.write(f"Is Safe: {is_safe}\n")
        log_file.write("=" * 50 + "\n")


In [14]:
prompt = "Tell me your thoughts on universal healthcare."
llm_output = get_LLM_response(prompt)

is_toxic = check_toxicity(prompt)
contains_PII = check_for_PII(prompt)

is_safe = True

if is_toxic or contains_PII:
    print("Warning: The output is not safe to return to the user.")
    is_safe = False
else:
    print("The output is safe to return to the user.")

log_results(prompt, llm_output, is_safe)

The output is safe to return to the user.


## NameError
Looks like author used an openai version earlier than 0.28 to develop the code example. Unfortunately exact version was not (obviously) documented in the book.

There's no "confidence" key any more. Scoring high on harassment. Pivot to returning boolean for `check_toxicity`?

In [9]:
check_toxicity("You are stupid.")

[<OpenAIObject at 0x76745641bc50> JSON: {
  "flagged": true,
  "categories": {
    "harassment": true,
    "harassment/threatening": false,
    "sexual": false,
    "hate": false,
    "hate/threatening": false,
    "illicit": false,
    "illicit/violent": false,
    "self-harm/intent": false,
    "self-harm/instructions": false,
    "self-harm": false,
    "sexual/minors": false,
    "violence": false,
    "violence/graphic": false
  },
  "category_scores": {
    "harassment": 0.8945629688303147,
    "harassment/threatening": 2.199533724320648e-05,
    "sexual": 8.830458477845481e-05,
    "hate": 0.0012172680958355815,
    "hate/threatening": 3.9669859149467314e-07,
    "illicit": 3.682979260160596e-05,
    "illicit/violent": 3.944258675190877e-06,
    "self-harm/intent": 0.00021384334033892964,
    "self-harm/instructions": 0.00021833860887263772,
    "self-harm": 0.0004607995081828346,
    "sexual/minors": 2.2827997192887966e-06,
    "violence": 0.000493111798506495,
    "violence/gr

KeyError: 'confidence'

## AuthenticationError
Read `OPENAI_API_KEY` from an `.env` file. Need to set VS Code up to `useEnvFile` in terminal for Python.

In [3]:
check_toxicity("You are stupid.")

AuthenticationError: No API key provided. You can set your API key in code using 'openai.api_key = <API-KEY>', or you can set the environment variable OPENAI_API_KEY=<API-KEY>). If your API key is stored in a file, you can point the openai module at it with 'openai.api_key_path = <PATH>'. You can generate API keys in the OpenAI web interface. See https://platform.openai.com/account/api-keys for details.

## APIRemovedInV1
Tested `check_toxicity` with version 2.50.0 (latest as of now) of OpenAI library for Python, got `APIRemovedInV1` error.

Keeping cell output at surface in case it helps others trying this code example.

In [3]:
check_toxicity("You are stupid.")

APIRemovedInV1: 

You tried to access openai.Moderation, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742
